# Indonesian ensemble rankings creation

This notebook creates a rankings file that ranks supplied tide models at specified coastal locations.
At each location, the NDWI inundation index from landsat and sentinal 2 satellite imagery is correlated against each tide
model, with the result used to rank the accuracy of each tide model.  A high correlation, i.e. a result that is
close to 1, has a higher ranking than a result that is closer to zero or a negative number.   

### Load packages

In [ ]:
import pandas as pd
import geopandas as gpd
from datacube.utils.dask import start_local_dask
from eo_tides.validation import tide_correlation

import os
os.environ["USE_PYGEOS"] = "0"

## Dask client
Create local dask client for parallelisation

In [ ]:
# Create local dask client for parallelisation
dask_client = start_local_dask(
    n_workers=4, threads_per_worker=8, mem_safety_margin="2GB"
)

dask_client

### Tide Models

Set model and model data location

In [ ]:
# for running in the sandbox
model_directory = "~/data/coastlines/tide_models/"

In [ ]:
# Check tide models
from eo_tides.utils import list_models

models = list_models(directory=model_directory, show_supported=False)
print(models)

In [ ]:
# Params
model_list =[
    'FES2014', 'FES2022', 'EOT20', 'TPXO9-atlas-v5-nc', 'TPXO10-atlas-v2-nc', 'GOT5.6', 'INATIDES'
]

### Points of interest

The points used for testing are determined following the process below and need to be extended to cover all locations along the Indonesian coastline.

1. Download https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/world-administrative-boundaries/exports/geojson
2. Extract Indonesian boundary.
3. Reproject Indonesian boundary to EPSG:32651 - WGS 84 / UTM zone 51N.
4. Convert polygon to linestring.
5. Create points along linestring every 10km
6. Clip to aoi (using 4km buffer of baseline coastline (2021))

In [ ]:
poi_file = "s3://files.auspatious.com/coastlines/indonesia_coastline_10km_points_aoi.geojson"

os.environ["AWS_DEFAULT_REGION"] = "ap-southeast-2"

In [ ]:
# Input tide ranking locations
poi = gpd.read_file(poi_file).to_crs('EPSG:4326')
coords = poi.geometry.get_coordinates()
len(coords)

### Tide rankings
For each point/location, correlate NDWI or NMDWI inundation with tide model and rank.

The data is loaded from datacube.

The processing will take many hours to go through 300 points.

In [ ]:
from odc.stac import configure_s3_access
from datacube import Datacube

dc = Datacube()

# disable landsat for now
# needs patching to work
load_landsat = False
if load_landsat:
    # Configure AWS for Landsat
    os.environ["AWS_DEFAULT_REGION"] = "us-west-2"
    if "AWS_NO_SIGN_REQUEST" in os.environ:
        del os.environ["AWS_NO_SIGN_REQUEST"]
    configure_s3_access(requester_pays=True)

In [ ]:
from odc.stac._mdtools import _normalize_geometry
from collections import Counter
import xarray as xr
import numpy as np
from odc.geo.geom import point

def mostcommon_crs(datasets):
    crs_counts = Counter(dataset.metadata_doc["crs"] for dataset in datasets)
    return crs_counts.most_common(1)[0][0]

def load_index(
    time = None,
    lon = None,
    lat = None,
    geopolygon = None,
    mask_geopolygon = False,
    crs = None,
    resolution = 30,
    resampling = "cubic",
    max_cloud_cover = 60,
    load_ls = load_landsat,
    load_s2 = True,
    chunks={"x": 2048, "y": 2048},
    index="ndwi",
):
    """Load an NDWI or MNDWI time-series from Landsat and/or Sentinel-2 from datacube.
    adapted from load_ndwi_mpc()

    Parameters
    ----------
    time : tuple, optional
        The time range to load data for as a tuple of strings (e.g.
        `("2020", "2021")`. If not provided, data will be loaded for
        all available timesteps.
    lon, lat : tuple, optional
        Tuples defining the spatial x and y extent to load in degrees.
    geopolygon : multiple types, optional
        Load data into the extents of a geometry. This could be an
        odc.geo Geometry, a GeoJSON dictionary, Shapely geometry, GeoPandas
        DataFrame or GeoSeries. GeoJSON and Shapely inputs are assumed to
        be in EPSG:4326 coordinates.
    mask_geopolygon : bool, optional
        Whether to mask pixels as nodata if they are outside the extent
        of a provided geopolygon. Defaults to False.
    crs : str, optional
        The Coordinate Reference System (CRS) to load data into. Defaults
        to None, which will attempt to load data into its most common native
        CRS to minimise resampling.
    resolution : int, optional
        Spatial resolution to load data in. Defaults to 30 metres.
    resampling : str, optional
        Resampling method used for surface reflectance bands. Defaults
        to "cubic"; "nearest" will always be used for categorical cloud
        masking bands.
    max_cloud_cover : int, optional
        The maximum threshold of cloud cover to load. Defaults to 60%.
    load_ls : bool, optional
        Whether to query and load Landsat data.
    load_s2 : bool, optional
        Whether to query and load Sentinel-2 data.
    chunks : dictionary, optional
        Dask chunking used to load data as lazy Dask backed arrays.
        Defaults to `{"x": 2048, "y": 2048}`.

    Returns
    -------
    satellite_ds : xarray.Dataset
        The loaded dataset as an `xarray.Dataset`, containing a single
        "ndwi" `xarray.DataArray`.

    """
    # Assemble parameters used for querying and loading
    query_params = {
        "time": time,
        "geopolygon": geopolygon,
    }
    load_params = {
        "resolution": resolution,
        "dask_chunks": chunks,
        "group_by": "solar_day",
        "resampling": {"qa_pixel": "nearest", "scl": "nearest", "*": resampling},
    }

    # List to hold outputs for each sensor (Landsat, Sentinel-2)
    output_list = []

    if load_ls:
        # Load Landsat
        datasets = dc.find_datasets(
            product=["ls8_c2l2_sr", "ls9_c2l2_sr"],
            cloud_cover=(0, max_cloud_cover),
            landsat_collection_category='T1',
            **query_params
        )
        print(f"Found {len(datasets)} Landsat datasets")
            
        if crs is None:
            crs = mostcommon_crs(datasets)

        if index == "mndwi":
            band2 = "swir16"
        elif index == "ndwi":
            band2 = "nir08"
        else:
            err_msg = "index can only be mndwi or ndwi"
            raise Exception(err_msg)
        
        ds_ls = dc.load(
            datasets=datasets,
            measurements=["green", band2, "qa_pixel"],
            output_crs=crs,
            **query_params,
            **load_params,
        )

        # Apply simple Landsat cloud mask
        cloud_mask = (
            # Bit 3: high confidence cloud, bit 4: high confidence shadow
            # https://medium.com/analytics-vidhya/python-for-geosciences-
            # raster-bit-masks-explained-step-by-step-8620ed27141e
            np.bitwise_and(ds_ls.qa_pixel, 1 << 3) | np.bitwise_and(
                ds_ls.qa_pixel, 1 << 4)
        ) == 0
        ds_ls = ds_ls.where(cloud_mask).drop_vars("qa_pixel")

        # Rescale to between 0.0 and 1.0
        ds_ls = (ds_ls.where(ds_ls != 0) * 0.0000275 + -0.2).clip(0, 1)

        # Convert to NDWI
        ndwi_ls = (ds_ls.green - ds_ls[band2]) / (ds_ls.green + ds_ls[band2])
        output_list.append(ndwi_ls)

    if load_s2:
        # Load Sentinel-2
        datasets = dc.find_datasets(
            product = ["s2_l2a"],
            cloud_cover=(0, max_cloud_cover),
            **query_params,
        )
        print(f"Found {len(datasets)} Sentinel-2 datasets")

        if crs is None:
            crs = mostcommon_crs(datasets)

        if index == "mndwi":
            band2 = "swir16"
        elif index == "ndwi":
            band2 = "nir"
        else:
            err_msg = "index can only be mndwi or ndwi"
            raise Exception(err_msg)
            
            
        ds_s2 = dc.load(
            datasets=datasets,
            measurements = ["green", band2, "scl"],
            output_crs=crs,
            driver="rio",
            **query_params,
            **load_params,
        )
        
        # Apply simple Sentinel-2 cloud mask
        # 1: defective, 3: shadow, 8:medium probability cloud, 9: high confidence cloud
        cloud_mask = ~ds_s2.scl.isin([1, 3, 8, 9])
        ds_s2 = ds_s2.where(cloud_mask).drop_vars("scl")

        # Rescale to between 0.0 and 1.0
        ds_s2 = (ds_s2.where(ds_s2 != 0) * 0.0001).clip(0, 1)

        # Convert to NDWI
        ndwi_s2 = (ds_s2.green - ds_s2[band2]) / (ds_s2.green + ds_s2[band2])
        output_list.append(ndwi_s2)

    # Merge into a single dataset
    ndwi = xr.concat(output_list, dim="time").sortby("time").to_dataset(name=index)

    # Optionally mask areas outside of supplied geopolygon (this has to be
    # applied here because applying it at the `stac_load` level converts
    # cloud masking bands to "float32".
    if mask_geopolygon & (geopolygon is not None):
        geopolygon = _normalize_geometry(geopolygon)
        ndwi = ndwi.odc.mask(poly=geopolygon)

    # return data for debug
    if load_ls and load_s2:
        return ndwi, ds_ls, ds_s2
    elif load_ls:
        return ndwi, ds_ls
    else:
        return ndwi, ds_s2

In [ ]:
out_list = []

# Loop through tide ranking locations and determine tide ranking
for index, row in coords.iterrows():

    print (f"Processing {index}, {row['x']}, {row['y']}")
    buffer =2500
    geom = point(x=row['x'], y=row['y'], crs="EPSG:4326").to_crs("utm").buffer(buffer).to_crs("EPSG:4326")

    index = "ndwi"
    # Load time series water_index (e.g. NDWI) for selected time period and location
    water_index, ds = load_index(
            time=("2023","2025"),
            geopolygon=geom,
            mask_geopolygon=False,
            max_cloud_cover=20,
            resolution=30,
            load_ls=False,
            load_s2=True,
            resampling= "bilinear",
            index=index,
        )
    water_index = water_index[index]

    corr_df, corr_da = tide_correlation(
        data=water_index,
        model=model_list,
        directory=model_directory,
    )

    out_list.append(corr_df)


### Data wrangling

Change the shape of the data to suit the tide model functions from eo-tides

The output file must include columns containing rankings for each tide model, named with the prefix "rank_". e.g. "rank_EOT20".
Low values should represent high rankings (e.g. 1 = top ranked).

It should also include a column named "valid_perc" for filtering.

In [ ]:
# Concatenate outputs and move "x", "y", "statistic" to columns
df_reset = pd.concat(out_list).reset_index()
#print(df_reset)

# --- rank columns: rank_{model} ---
rank_wide = (
    df_reset.pivot(index=["x", "y"], columns="tide_model", values="rank")
    .rename(columns=lambda m: f"rank_{m}")
)

# --- correlation columns: corr_{model} ---
corr_wide = (
    df_reset.pivot(index=["x", "y"], columns="tide_model", values="correlation")
    .rename(columns=lambda m: f"corr_{m}")
)

# --- top and worst model per x,y ---
idx_top = df_reset.groupby(["x", "y"])["correlation"].idxmax()
idx_worst = df_reset.groupby(["x", "y"])["correlation"].idxmin()

top_model = (
    df_reset.loc[idx_top, ["x", "y", "tide_model", "correlation", "valid_perc"]]
    .rename(columns={
        "tide_model": "top_model",     # best model
        "correlation": "top_correlation",     # value used for ranking
        "rank_valid_perc": "valid_perc" # single valid_perc
    })
    .set_index(["x", "y"])
)

worst_model = (
    df_reset.loc[idx_worst, ["x", "y", "tide_model", "correlation"]]
    .rename(columns={
        "tide_model": "worst_model",
        "correlation": "worst_correlation"
    })
    .set_index(["x", "y"])
)

# --- combine everything ---
out = pd.concat(
    [rank_wide, corr_wide, top_model, worst_model],
    axis=1
).reset_index()

# --- add geometry ---
model_rankings_gdf = gpd.GeoDataFrame(
    out,
    geometry=gpd.points_from_xy(out["x"], out["y"]),
    crs="EPSG:4326",
)

In [ ]:
df_reset.groupby('tide_model').describe()['rank']

### Output
Save the result as a flatgeobuf file

In [ ]:
model_rankings_gdf.to_file ('indo_model_ranking_s2.fgb', driver='FlatGeobuf')

### Close Dask client

In [ ]:
dask_client.close()